# Kalaratri: which log sessions are failing?

LogHub HDFS, attached from its public Kaggle copy (research use; not redistributed here). Each block is a session of log events; its label says whether it failed. Train on the earlier 80% of sessions and test on the later 20%, so the model is tested on the future, never on a shuffled past.

Kalaratri is one of the nine models of **Sati**, the investigation agent of [Pashupatastra](https://github.com/preetraval45/PASHUPATASTRA), each named for a form of Navadurga. This notebook reproduces its core result on the public dataset, with the split that keeps the test honest.

In [ ]:
import glob, os
def find(pattern):
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    assert hits, f"{pattern} not found under /kaggle/input"
    return hits[0]
import pandas as pd, numpy as np
m = pd.read_csv(find("Event_occurrence_matrix.csv"))
label = next(c for c in m.columns if c.lower() == "label")
m["anomaly"] = m[label].astype(str).str.lower().str.contains("fail|anomal").astype(int)
events = [c for c in m.columns if c.startswith("E") and c[1:].isdigit()]
print(m.shape, "anomaly share", round(m["anomaly"].mean(), 4))

In [ ]:
import lightgbm as lgb
from sklearn.metrics import f1_score, precision_score, recall_score
cut = int(len(m) * 0.8)
X, y = np.log1p(m[events]), m["anomaly"]
model = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, verbose=-1).fit(X[:cut], y[:cut])
pred = model.predict(X[cut:])
print(f"precision {precision_score(y[cut:], pred):.3f}  recall {recall_score(y[cut:], pred):.3f}  F1 {f1_score(y[cut:], pred):.3f}")

In [ ]:
from sklearn.decomposition import PCA
normal = X[:cut][y[:cut] == 0]
pca = PCA(n_components=0.95).fit(normal)
err = ((X[cut:] - pca.inverse_transform(pca.transform(X[cut:]))) ** 2).sum(axis=1)
threshold = np.quantile(((normal - pca.inverse_transform(pca.transform(normal))) ** 2).sum(axis=1), 0.995)
print(f"PCA baseline F1: {f1_score(y[cut:], err > threshold):.3f}")

**Reading it:** on HDFS, event counts separate failing sessions almost perfectly; a plain model on this copy's row order scores about 0.999, and beats the classic PCA detector (Xu et al., 2009). This copy's rows are not strictly in time order, so the split is easier than a true time split: Kalaratri, split by each session's first timestamp, reaches 0.966. On BGL, where failures are rarer and less templated, it reaches only 0.355. HDFS alone flatters log anomaly detection.